# Autoregressive language model (`variant: autoregressive`)

<img src="../../assets/explainers/autoregressive.gif" width="100%">

The animation above is generated by this notebook's code from a real (tiny) trained model. **Training**: the target row is the input shifted by one - every position learns the next character, all 47 lessons in one forward pass. **Generation**: the model box lights up once per forward pass, the real next-character distribution grows out of it, one character is sampled and flies into its slot (cyan underlines = attention of the new position). The two 50/50 branch points are held in slow motion; the counter shows one forward pass per character.

**Contents:** intuition → math → where it lives in the code → train a toy → teacher forcing and the causal
mask, hands-on → the branch points of the corpus → temperature → step-by-step trace → takeaways.

## Intuition

Read left to right and, at every position, guess the next token. If every guess is a proper probability
distribution, multiplying the guesses gives the probability of the whole sequence - that is the chain
rule, no approximation involved. Generation replays the same game: guess, commit, append, repeat.

## Math

$$p_\theta(x_{1:N}) = \prod_{i=1}^{N} p_\theta(x_i \mid x_{<i}), \qquad
  \mathcal{L}(\theta) = -\frac{1}{N}\sum_{i} \log p_\theta(x_{i+1} \mid x_{\le i}).$$

* **Teacher forcing.** In training the model always conditions on the *true* prefix, so all $N$
  predictions of a sequence are computed in **one** forward pass.
* **Causal mask.** Attention from query $i$ to key $j$ is allowed only if $j \le i$
  ($M_{ij} = -\infty$ for $j > i$), so position $i$ cannot peek at the answer $x_{i+1}$.
* **Sampling.** $x_{i+1} \sim \mathrm{softmax}(z_i / T)$ with temperature $T$; $T \to 0$ is greedy.
  Generating $N$ tokens costs $N$ sequential forward passes.

## In the code

| File | What |
|---|---|
| `src/hale_llm/models/models_autoregressive/model.py` | `AutoregressiveLM` + `collate_autoregressive` (labels = inputs shifted by one, pads → -100) |
| `.../loss.py` | `autoregressive_loss`: token NLL on `logits[:, :-1]` vs `labels[:, :-1]` |
| `.../sample.py` | `sample_autoregressive`: crop to `max_length`, sample, append |
| `hale_core.nn.layers.attention.build_attn_mask("causal", …)` | the mask (HaleBlocks) |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Any backbone works (`model.arch`); the toy uses the default GPT stack. The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("autoregressive")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## Teacher forcing: what one batch looks like

`collate_autoregressive` builds the targets by shifting: `labels[i] = input_ids[i + 1]`.

In [ ]:
batch = get_collate("autoregressive")(run.ids, tok)
print("input :", "".join(tok.token(i) for i in batch["input_ids"][0][:24]))
print("labels:", "".join(tok.token(i) if i >= 0 else "∅" for i in batch["labels"][0][:24]))
with torch.no_grad():
    print("loss on the sentence:", float(get_loss("autoregressive")(model, batch)))

In [ ]:
from hale_core.nn.layers.attention import build_attn_mask

mask = build_attn_mask("causal", 12, "cpu")
fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.imshow(~mask, cmap="Blues")
labels = list(run.text[:12].replace(" ", "␣"))
ax.set_xticks(range(12), labels); ax.set_yticks(range(12), labels)
ax.set(xlabel="key (attended to)", ylabel="query", title="causal mask: blue = allowed")
plt.show()

## The branch points

The corpus has two binary choices: "model" vs "robot" right after the prompt and "write" vs "spell"
later. A well-trained model should put ~50/50 mass on the two continuations exactly there and be
near-certain everywhere else.

In [ ]:
def next_char_probs(prefix: str, k: int = 4):
    ids = torch.tensor([tok.encode(prefix)])
    with torch.no_grad():
        probs = F.softmax(model({"input_ids": ids})[0, -1], -1)
    top = probs.topk(k)
    return {tok.token(i): round(float(p), 3) for i, p in zip(top.indices, top.values)}

for prefix in ["the tiny ", "the tiny m", "the tiny model learned to ", "the tiny model learned to w"]:
    print(f"{prefix!r:32} -> {next_char_probs(prefix)}")

## Temperature

Dividing the logits by $T$ sharpens ($T<1$) or flattens ($T>1$) every distribution. With this model the
branch points are the only places where it matters - higher $T$ eventually produces characters that are
in no training sentence.

In [ ]:
sampler = get_sampler("autoregressive")
for T in (0.01, 0.7, 1.5, 3.0):
    torch.manual_seed(1)
    outs = [tok.decode(sampler(model, tok, run.ids[:, :8], max_new_tokens=40, temperature=T)[0])
            for _ in range(6)]
    valid = sum(o in CORPUS for o in outs)
    print(f"T={T:<5} valid {valid}/6   e.g. {outs[0]!r}")

## Generation, step by step

`explainer.trace` mirrors `sample_autoregressive` operation for operation (the unit tests check that the
two produce identical tokens for the same seed) and records the distribution at every step.

In [ ]:
trace = explainer.trace(run, seed=0)
assert torch.equal(trace.ids, explainer.reference(run, seed=0))
for k, step in enumerate(trace.steps[:12]):
    top = ", ".join(f"{tok.token(i)!r}:{p:.2f}" for i, p in zip(step.top_ids[:3], step.top_probs[:3]))
    print(f"step {k + 1:2d}  sampled {tok.token(step.chosen)!r}  p={step.chosen_prob:.2f}   top-3 [{top}]")
print("…\noutput:", tok.decode(trace.ids[0]))

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/autoregressive.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "autoregressive.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/autoregressive.gif" width="100%">'))

## Takeaways

* Exact likelihood and the simplest possible training signal - every position is supervised in parallel.
* Inference is inherently sequential: $N$ tokens = $N$ forward passes (a KV cache makes each pass cheap,
  but not parallel). MTP (`02_mtp`) and the diffusion family (`04`–`07`) attack exactly this.
* Every committed token is final: there is no way to revise an early mistake.

**Exercises.** (1) Train with `train_kwargs={"steps": 30}` and watch the branch-point distributions
before convergence. (2) Swap the backbone: `get_explainer("autoregressive")` with
`train_toy("autoregressive", CORPUS, model={"arch": "lgt", "attn_type": "causal"})`.